In [1]:
from google.colab import drive
drive.mount('/content/drive')
import sys, importlib, re, numpy as np, pandas as pd, torch
from scipy.stats import wilcoxon
CODE = "/content/drive/MyDrive/esci/code"; sys.path.insert(0, CODE)
import metrics, cross_encoder as C, train_cross_encoder as T
for m in (metrics, C, T): importlib.reload(m)
from metrics import evaluate
BASE = "/content/drive/MyDrive/esci"

df = pd.read_parquet(f"{BASE}/processed/rerank_pairs_v2.3.parquet")
qs = pd.read_parquet(f"{BASE}/dataset_v2.3/esci_electronics_v2.3_queries.parquet")[["query_id", "rule", "scope_group"]]
df = df.merge(qs, on="query_id", how="left")
ids = {p: set(df.loc[df.part == p, "query_id"]) for p in ["train", "val", "test"]}
assert not (ids["train"] & ids["val"]) and not (ids["train"] & ids["test"]) and not (ids["val"] & ids["test"])
train = df[df.part == "train"].reset_index(drop=True)
val   = df[df.part == "val"].reset_index(drop=True)
del df                                                        # test rows dropped

CAT = {"mobile_computing": "Mobile & Computing", "components_accessories": "Components & Accessories",
       "home_electronics": "Home Electronics", "imaging": "Cameras & Imaging", "audio": "Audio",
       "computer_peripherals": "Computer Peripherals", "gaming": "Gaming", "smart_home": "Smart Home"}
def first_words(t, n): return " ".join(C._clean(t).split()[:n])
def drop_brand(t, b):
    t, b = C._clean(t), C._clean(b)
    out = " ".join(re.sub(re.escape(b), " ", t, flags=re.I).split()) if b else t
    return out or t
def kv(r, with_title=False, bullets=False):
    parts = [f"brand: {C._clean(r['product_brand'])}" if C._clean(r['product_brand']) else "",
             f"color: {C._clean(r['product_color'])}" if C._clean(r['product_color']) else ""]
    if bullets and C._clean(r['product_bullet_point']): parts.append("features: " + C._clean(r['product_bullet_point'])[:200])
    if with_title: parts = parts + [C._clean(r['product_title'])]
    return " | ".join(x for x in parts if x)

for d in (train, val):
    d["category"] = np.where(d["rule"] == "A", d["scope_group"].map(CAT), None)
    d["q_focused_cat"] = [C.query_to_text(q, category=c) for q, c in zip(d["query"], d["category"])]
    prods = [C.esci_row_to_product(r) for r in d.to_dict("records")]
    for s in C.STYLES:
        d[f"text_{s}"] = [C.product_to_text(p, s) for p in prods]

# ---- TRAINING-ONLY harder formats (random but fixed: seed 7)
rng = np.random.default_rng(7)
recs = train.to_dict("records")
train["text_aug_short"]    = [first_words(r["product_title"], int(rng.integers(3, 9))) for r in recs]   # 3-8 words
train["text_aug_no_brand"] = [drop_brand(r["product_title"], r["product_brand"]) for r in recs]
def shuffled(r):
    parts = [C._clean(r["product_title"]),
             f"brand: {C._clean(r['product_brand'])}" if C._clean(r['product_brand']) else "",
             f"color: {C._clean(r['product_color'])}" if C._clean(r['product_color']) else ""]
    parts = [p for p in parts if p]; rng.shuffle(parts); return " | ".join(parts)
train["text_aug_shuffled"] = [shuffled(r) for r in recs]

# ---- VALIDATION-ONLY hard formats (exactly as in S9)
HARD = {"title_4words":   lambda r: first_words(r["product_title"], 4),
        "title_no_brand": lambda r: drop_brand(r["product_title"], r["product_brand"]),
        "title_noisy":    lambda r: " ".join(re.sub(r"[^a-z0-9 ]", " ", C._clean(r["product_title"]).lower()).split()),
        "kv_title_last":  lambda r: kv(r, with_title=True),
        "no_title":       lambda r: kv(r, bullets=True)}
vrecs = val.to_dict("records")
for n, f in HARD.items():
    val[f"text_{n}"] = [f(r) for r in vrecs]

TRAIN_STYLES = C.STYLES + ["aug_short", "aug_no_brand", "aug_shuffled"]
EVAL_STYLES  = C.STYLES + list(HARD)
SEEN_TYPE, UNSEEN_TYPE = ["title_4words", "title_no_brand", "kv_title_last"], ["title_noisy", "no_title"]
print("train", len(train), "| val", len(val), "| GPU", torch.cuda.get_device_name(0))
print("train formats:", TRAIN_STYLES); print("eval formats:", EVAL_STYLES)
print(train[["text_title", "text_aug_short", "text_aug_no_brand", "text_aug_shuffled"]].head(3).to_string())

Mounted at /content/drive
train 65852 | val 11741 | GPU Tesla T4
train formats: ['title', 'title_brand', 'full', 'full_bullets', 'key_value', 'aug_short', 'aug_no_brand', 'aug_shuffled']
eval formats: ['title', 'title_brand', 'full', 'full_bullets', 'key_value', 'title_4words', 'title_no_brand', 'title_noisy', 'kv_title_last', 'no_title']
                                                                                                                                                                                             text_title                                         text_aug_short                                                                                                                                                                              text_aug_no_brand                                                                                                                                                                                                                    te

In [2]:
LOG_AUG = "s9_aug_val_log.csv"
for seed in (42, 43, 44):
    cfg = {**T.CFG, "query_col": "q_focused_cat", "styles": TRAIN_STYLES, "eval_styles": EVAL_STYLES, "seed": seed}
    name = "minilm_l6_aug__q_focused_cat" + ("" if seed == 42 else f"__s{seed}")
    print(f"\n===== {name} =====")
    T.train_run("cross-encoder/ms-marco-MiniLM-L-6-v2", name, train, val, evaluate, BASE, cfg=cfg, log_name=LOG_AUG)


===== minilm_l6_aug__q_focused_cat =====
minilm_l6_aug__q_focused_cat: starting at epoch 1 from cross-encoder/ms-marco-MiniLM-L-6-v2


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

...

epoch 1 done: loss 0.5077, 1.6 min
         style  ndcg@10    mrr    p@5
         title   0.9228 0.9414 0.8460
   title_brand   0.9236 0.9413 0.8454
          full   0.9245 0.9400 0.8470
  full_bullets   0.9276 0.9430 0.8476
     key_value   0.9245 0.9416 0.8442
  title_4words   0.8760 0.8961 0.7951
title_no_brand   0.9028 0.9139 0.8213
   title_noisy   0.9231 0.9413 0.8430
 kv_title_last   0.9243 0.9420 0.8439
      no_title   0.8449 0.8766 0.7613
  epoch 2 step 500/2058 loss 0.3687
  epoch 2 step 1000/2058 loss 0.3642
  epoch 2 step 1500/2058 loss 0.3605
  epoch 2 step 2000/2058 loss 0.3564


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

epoch 2 done: loss 0.3561, 1.6 min
         style  ndcg@10    mrr    p@5
         title   0.9262 0.9438 0.8466
   title_brand   0.9273 0.9428 0.8488
          full   0.9276 0.9453 0.8506
  full_bullets   0.9295 0.9403 0.8518
     key_value   0.9303 0.9469 0.8466
  title_4words   0.8776 0.8937 0.7896
title_no_brand   0.9052 0.9169 0.8207
   title_noisy   0.9221 0.9391 0.8442
 kv_title_last   0.9268 0.9448 0.8460
      no_title   0.8494 0.8768 0.7652
  epoch 3 step 500/2058 loss 0.3288
  epoch 3 step 1000/2058 loss 0.3300
  epoch 3 step 1500/2058 loss 0.3293
  epoch 3 step 2000/2058 loss 0.3270


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

epoch 3 done: loss 0.3262, 1.6 min
         style  ndcg@10    mrr    p@5
         title   0.9245 0.9414 0.8476
   title_brand   0.9258 0.9425 0.8488
          full   0.9276 0.9450 0.8488
  full_bullets   0.9274 0.9402 0.8497
     key_value   0.9283 0.9448 0.8482
  title_4words   0.8767 0.8937 0.7918
title_no_brand   0.9043 0.9141 0.8192
   title_noisy   0.9225 0.9423 0.8436
 kv_title_last   0.9258 0.9450 0.8460
      no_title   0.8486 0.8746 0.7674


In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
LOG_AUG = "s9_aug_val_log.csv"
def best_epochs(log_file, prefix):            # epoch chosen on NORMAL formats only
    L = pd.read_csv(f"{BASE}/results/{log_file}")
    L = L[L.run.str.startswith(prefix + "__q_focused_cat") & L["style"].isin(C.STYLES)]
    ep = L.groupby(["run", "epoch"])["ndcg@10"].mean().reset_index()
    return ep.loc[ep.groupby("run")["ndcg@10"].idxmax()][["run", "epoch"]]
GROUPS = {"mix": best_epochs("s7_grid_val_log.csv", "minilm_l6"), "aug": best_epochs(LOG_AUG, "minilm_l6_aug")}
for g, b in GROUPS.items(): print(g, b.values.tolist())

PQ, rows, qtext = {}, [], val["q_focused_cat"].tolist()
for g, b in GROUPS.items():
    for _, r in b.iterrows():
        path = f"{BASE}/models/{r.run}/epoch_{int(r.epoch)}"
        tok = AutoTokenizer.from_pretrained(path)
        m = AutoModelForSequenceClassification.from_pretrained(path).to("cuda").eval()
        for f in EVAL_STYLES:
            v = val[["query_id", "gain"]].copy()
            v["score"] = T.score_pairs(m, tok, qtext, val[f"text_{f}"].tolist(), 256, 128)
            mtr, pq = evaluate(v); rows.append({"group": g, "run": r.run, "format": f, **mtr})
            PQ[(g, f)] = PQ.get((g, f), []) + [pq.set_index("query_id")["ndcg@10"]]
        del m; torch.cuda.empty_cache(); print("scored", r.run)

R = pd.DataFrame(rows)
tab = R.groupby(["format", "group"])["ndcg@10"].agg(["mean", "std"]).unstack("group").reindex(EVAL_STYLES)
tab[("diff", "aug-mix")] = tab[("mean", "aug")] - tab[("mean", "mix")]
print(tab.round(4).to_string())

def avg(g, formats):            # per-query NDCG averaged over formats and seeds
    return pd.concat([pd.concat(PQ[(g, f)], axis=1).mean(axis=1) for f in formats], axis=1).mean(axis=1)
print()
for label, fm in [("NORMAL (5)", C.STYLES), ("HARD (5)", list(HARD)), ("hard seen-type", SEEN_TYPE), ("hard UNSEEN-type", UNSEEN_TYPE)] + [(f, [f]) for f in HARD]:
    a, b = avg("aug", fm), avg("mix", fm); d = a - b.loc[a.index]; nz = d[d != 0]
    print(f"{label:17s}: aug - mix = {d.mean():+.4f} | better {int((d>0).sum())} / worse {int((d<0).sum())} | p = {wilcoxon(nz).pvalue:.4f}")

# ---- decision rule (fixed in advance)
dh = avg("aug", list(HARD)) - avg("mix", list(HARD)); dn = avg("aug", C.STYLES) - avg("mix", C.STYLES)
p_h = wilcoxon(dh[dh != 0]).pvalue; p_n = wilcoxon(dn[dn != 0]).pvalue
ok_hard = dh.mean() > 0 and p_h < 0.05
ok_norm = dn.mean() >= 0 or p_n >= 0.05
print(f"\nhard better & significant: {ok_hard} | normal not significantly worse: {ok_norm}")
print("DECISION:", "use AUGMENTED model as final" if (ok_hard and ok_norm) else "keep CURRENT (mix) model")
R.to_csv(f"{BASE}/results/s9_aug_vs_mix.csv", index=False)

mix [['minilm_l6__q_focused_cat', 3], ['minilm_l6__q_focused_cat__s43', 3], ['minilm_l6__q_focused_cat__s44', 2]]
aug [['minilm_l6_aug__q_focused_cat', 1], ['minilm_l6_aug__q_focused_cat__s43', 1], ['minilm_l6_aug__q_focused_cat__s44', 2]]


Loading weights:   0%|          | 0/105 [00:01<?, ?it/s]

scored minilm_l6__q_focused_cat


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

scored minilm_l6__q_focused_cat__s43


...

scored minilm_l6_aug__q_focused_cat


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

scored minilm_l6_aug__q_focused_cat__s43


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

scored minilm_l6_aug__q_focused_cat__s44
                  mean             std            diff
group              aug     mix     aug     mix aug-mix
format                                                
title           0.9270  0.9270  0.0012  0.0011  0.0000
title_brand     0.9269  0.9281  0.0005  0.0009 -0.0011
full            0.9273  0.9278  0.0003  0.0012 -0.0005
full_bullets    0.9284  0.9286  0.0013  0.0012 -0.0002
key_value       0.9287  0.9280  0.0014  0.0011  0.0007
title_4words    0.8780  0.8811  0.0006  0.0007 -0.0031
title_no_brand  0.9056  0.9008  0.0008  0.0018  0.0048
title_noisy     0.9236  0.9239  0.0017  0.0010 -0.0004
kv_title_last   0.9265  0.9270  0.0006  0.0025 -0.0005
no_title        0.8486  0.8469  0.0035  0.0007  0.0017

NORMAL (5)       : aug - mix = -0.0002 | better 150 / worse 161 | p = 0.5208
HARD (5)         : aug - mix = +0.0005 | better 228 / worse 206 | p = 0.5454
hard seen-type   : aug - mix = +0.0004 | better 193 / worse 192 | p = 0.7431
hard UNSEEN-